<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-12-production-deploy/lesson-12.1-infra-setup/practice/GCP_Capstone_12.1_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 12.1 — Infrastructure & Project Setup

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

> ⚠️ **This lab provisions live infrastructure (Terraform + gcloud).** Run cells in exercise order and do **not** use *Run all*. Exercise 8's `terraform apply` creates the Pub/Sub topic and Secret Manager secrets that the Exercise 7 and 8 verification cells depend on — apply the Terraform first, then run those checks.

## Setup: auth, tooling, and project variables

Run these three cells first. They authenticate with Application Default Credentials, install Terraform into the Colab VM, and set the project variables every exercise below depends on. Change `PROJECT_ID` to your own project before running.

> All `gcloud`/`gsutil`/`terraform` shell cells read the environment variables exported here, so the Python cell must run before any bash cell.

In [ ]:
# Application Default Credentials (never API keys) + project variables
from google.colab import auth
auth.authenticate_user()

import os

PROJECT_ID   = 'documind-ai-YOUR-ID'  # CHANGE THIS
REGION       = 'us-central1'          # course examples
INDIA_REGION = 'asia-south1'          # DPDPA-aligned India prod
USD_INR      = 85                     # cost-display conversion

# Export so the %%bash cells below can see them
os.environ['PROJECT_ID']   = PROJECT_ID
os.environ['REGION']       = REGION
os.environ['INDIA_REGION'] = INDIA_REGION

!gcloud config set project {PROJECT_ID}
print('Project set to', PROJECT_ID)

In [ ]:
%%bash
# Install a pinned Terraform (>= 1.9) into the Colab VM
set -e
TF_VERSION=1.9.8
curl -fsSL -o /tmp/tf.zip "https://releases.hashicorp.com/terraform/${TF_VERSION}/terraform_${TF_VERSION}_linux_amd64.zip"
unzip -o /tmp/tf.zip -d /usr/local/bin/ >/dev/null
terraform version

## Exercise 1: Bootstrap project + APIs

**Difficulty:** Easy

Create the project, set billing, enable all the capstone APIs with a single `gcloud services enable`. Create the `documind-tfstate` GCS bucket with versioning so Terraform state is safe.

1. Point `gcloud` at the project.
2. Enable every API the capstone needs in one command.
3. Create the Terraform state bucket and turn on object versioning.

**Expected behaviour:** `gcloud services list` shows the APIs enabled. `gsutil ls -b gs://documind-tfstate` returns `Versioning=Enabled`.

In [ ]:
%%bash
set -e
gcloud config set project "$PROJECT_ID"

# One gcloud command, every API the capstone needs
gcloud services enable \
  run.googleapis.com \
  compute.googleapis.com \
  vpcaccess.googleapis.com \
  pubsub.googleapis.com \
  eventarc.googleapis.com \
  cloudfunctions.googleapis.com \
  artifactregistry.googleapis.com \
  secretmanager.googleapis.com \
  firestore.googleapis.com \
  storage.googleapis.com \
  aiplatform.googleapis.com \
  documentai.googleapis.com \
  speech.googleapis.com \
  texttospeech.googleapis.com \
  dlp.googleapis.com \
  iap.googleapis.com \
  iamcredentials.googleapis.com \
  cloudbuild.googleapis.com \
  cloudtrace.googleapis.com \
  monitoring.googleapis.com \
  logging.googleapis.com \
  billingbudgets.googleapis.com

echo '--- APIs enabled ---'
gcloud services list --enabled --filter='config.name~(run|firestore|secretmanager|artifactregistry)' --format='value(config.name)'

In [ ]:
%%bash
set -e
# Terraform state bucket. backend.tf (Exercise 2) hard-codes this exact name.
gsutil mb -l "$REGION" -b on "gs://${PROJECT_ID}-tfstate" || echo 'bucket already exists, continuing'
gsutil versioning set on "gs://${PROJECT_ID}-tfstate"

echo '--- versioning status ---'
gsutil versioning get "gs://${PROJECT_ID}-tfstate"
# NOTE: do NOT lock a retention policy on the state bucket -- locked retention
# would prevent Terraform from ever pruning old state versions. Locked retention
# is reserved for the audit bucket (Exercise 6).

## Exercise 2: backend.tf + provider.tf + first plan

**Difficulty:** Easy

Write `backend.tf` with the GCS backend, pin Terraform >= 1.9 and the google provider ~> 6.15. Run `terraform init` then a first `terraform plan`.

1. Write `backend.tf` (backend + required providers + provider blocks).
2. Run `terraform init` against the GCS backend.
3. Run `terraform plan` and verify a clean exit.

**Expected behaviour:** `terraform init` succeeds and writes state/lock to GCS. The plan runs without provider errors.

In [ ]:
BACKEND_TF = '''
terraform {
  required_version = ">= 1.9.0"
  required_providers {
    google      = { source = "hashicorp/google",      version = "~> 6.15" }
    google-beta = { source = "hashicorp/google-beta", version = "~> 6.15" }
  }
  backend "gcs" {
    bucket = "documind-tfstate"
    prefix = "documind/env"
  }
}

provider "google" {
  project = var.project_id
  region  = var.region
}
provider "google-beta" {
  project = var.project_id
  region  = var.region
}
'''
with open('backend.tf', 'w') as f:
    f.write(BACKEND_TF)
print('backend.tf written')

In [ ]:
%%bash
set -e
terraform init -reconfigure
# First plan. variables.tf lands in Exercise 3, so pass the required vars inline.
terraform plan \
  -var=project_id="$PROJECT_ID" \
  -var=region="$REGION" \
  -var='admin_emails=["alice@documind.ai"]' \
  -var=billing_account_id=YOUR-BILLING-ID \
  || echo 'plan needs the .tf files from the next exercises -- re-run after Exercise 8'

## Exercise 3: Three service accounts with least-privilege roles

**Difficulty:** Medium

Create `documind-ui-sa`, `documind-api-sa`, `documind-admin-sa`. Attach scoped role lists via `for_each`. Add the self-impersonation binding on `ui-sa` (needed for V4 signed URLs on Cloud Run). Apply.

1. Write `variables.tf` with the project/region/env inputs.
2. Write `sa.tf`: three service accounts, per-SA scoped role locals, `for_each` project bindings.
3. Grant `ui-sa` the token-creator role on itself.

**Expected behaviour:** `gcloud iam service-accounts list` shows 3 SAs. `ui-sa` holds `roles/iam.serviceAccountTokenCreator` on itself.

In [ ]:
VARS_TF = '''
variable "project_id"  { type = string }
variable "region"      {
  type    = string
  default = "us-central1"
}
variable "india_region"{
  type    = string
  default = "asia-south1"
}
variable "env"         {
  type    = string
  default = "dev"
}  # dev | staging | prod
variable "admin_emails"{ type = list(string) }
'''

SA_TF = '''
# Three service accounts, one per service, least-privilege from day zero.
resource "google_service_account" "ui" {
  account_id   = "documind-ui-sa"
  display_name = "DocuMind UI (Streamlit on Cloud Run)"
}
resource "google_service_account" "api" {
  account_id   = "documind-api-sa"
  display_name = "DocuMind API (FastAPI backend)"
}
resource "google_service_account" "admin" {
  account_id   = "documind-admin-sa"
  display_name = "DocuMind Admin + Observability"
}

# Signed URLs on Cloud Run need self-impersonation
resource "google_service_account_iam_member" "ui_self_impersonate" {
  service_account_id = google_service_account.ui.name
  role               = "roles/iam.serviceAccountTokenCreator"
  member             = "serviceAccount:${google_service_account.ui.email}"
}

locals {
  ui_roles = [
    "roles/aiplatform.user",
    "roles/documentai.apiUser",
    "roles/secretmanager.secretAccessor",
    "roles/datastore.user",
    "roles/speech.editor",
  ]
  api_roles = [
    "roles/aiplatform.user",
    "roles/datastore.user",
    "roles/secretmanager.secretAccessor",
    "roles/logging.logWriter",
    "roles/cloudtrace.agent",
  ]
  admin_roles = [
    "roles/monitoring.viewer",
    "roles/logging.viewer",
    "roles/datastore.user",
    "roles/bigquery.jobUser",
    "roles/run.developer",   # budget-guard fn (Exercise 8) calls run_v2.update_service -> min_instances=0
  ]
}

resource "google_project_iam_member" "ui" {
  for_each = toset(local.ui_roles)
  project  = var.project_id
  role     = each.value
  member   = "serviceAccount:${google_service_account.ui.email}"
}
resource "google_project_iam_member" "api" {
  for_each = toset(local.api_roles)
  project  = var.project_id
  role     = each.value
  member   = "serviceAccount:${google_service_account.api.email}"
}
resource "google_project_iam_member" "admin" {
  for_each = toset(local.admin_roles)
  project  = var.project_id
  role     = each.value
  member   = "serviceAccount:${google_service_account.admin.email}"
}
'''
with open('variables.tf', 'w') as f: f.write(VARS_TF)
with open('sa.tf', 'w') as f: f.write(SA_TF)
print('variables.tf + sa.tf written')

## Exercise 4: VPC + connector + Artifact Registry

**Difficulty:** Medium

Provision `documind-vpc` with a /20 subnet + VPC Access Connector (min=2 max=6) for private egress. Create the Artifact Registry repo with a keep-recent-20 + delete-older-than-30-day cleanup policy.

1. Write `network.tf`: VPC, /20 subnet with private Google access, VPC Access Connector.
2. Write `registry.tf`: Docker repo with both cleanup policies.

**Expected behaviour:** `gcloud compute networks describe documind-vpc` succeeds. The repo shows both cleanup policies in its JSON.

In [ ]:
NETWORK_TF = '''
resource "google_compute_network" "vpc" {
  name                    = "documind-vpc"
  auto_create_subnetworks = false
}
resource "google_compute_subnetwork" "subnet" {
  name          = "documind-subnet"
  network       = google_compute_network.vpc.id
  region        = var.region
  ip_cidr_range = "10.20.0.0/20"
  private_ip_google_access = true
}
resource "google_vpc_access_connector" "conn" {
  name          = "documind-vpc"
  region        = var.region
  network       = google_compute_network.vpc.name
  ip_cidr_range = "10.8.0.0/28"
  min_instances = 2
  max_instances = 6
}
'''

REGISTRY_TF = '''
resource "google_artifact_registry_repository" "docker" {
  repository_id = "documind"
  format        = "DOCKER"
  location      = var.region
  description   = "DocuMind container images"
  cleanup_policies {
    id     = "keep-recent"
    action = "KEEP"
    most_recent_versions { keep_count = 20 }
  }
  cleanup_policies {
    id     = "delete-old"
    action = "DELETE"
    condition { older_than = "2592000s" }  # 30 days
  }
}
'''
with open('network.tf', 'w') as f: f.write(NETWORK_TF)
with open('registry.tf', 'w') as f: f.write(REGISTRY_TF)
print('network.tf + registry.tf written')

## Exercise 5: Firestore asia-south1 with PITR + delete protection

**Difficulty:** Medium

Create the `(default)` Firestore database in `asia-south1`, Native mode, with point-in-time recovery and delete protection both enabled. Delete protection means a `terraform destroy` on the resource is refused.

1. Write `firestore.tf` in `asia-south1` (DPDPA data residency).
2. Enable PITR and delete protection.

**Expected behaviour:** `gcloud firestore databases describe` shows PITR + protection enabled. A destroy fails with a delete-protection error.

In [ ]:
FIRESTORE_TF = '''
resource "google_firestore_database" "main" {
  project     = var.project_id
  name        = "(default)"
  location_id = var.india_region   # DPDPA-aligned data residency
  type        = "FIRESTORE_NATIVE"
  concurrency_mode                = "OPTIMISTIC"
  app_engine_integration_mode     = "DISABLED"
  point_in_time_recovery_enablement = "POINT_IN_TIME_RECOVERY_ENABLED"
  delete_protection_state         = "DELETE_PROTECTION_ENABLED"
}
'''
with open('firestore.tf', 'w') as f: f.write(FIRESTORE_TF)
print('firestore.tf written')
print()
print('After apply, prove delete protection is on -- this destroy is expected to FAIL:')
print('  terraform destroy -target=google_firestore_database.main \\')
print('    -var=project_id=$PROJECT_ID -var=india_region=$INDIA_REGION')

## Exercise 6: Three GCS buckets with lifecycle + locked retention

**Difficulty:** Medium

Provision `uploads` (90d -> NEARLINE, 365d delete, CORS), `tts-cache` (30d delete), and `audit` (5y LOCKED retention). Bind `ui-sa` as `objectAdmin` on uploads + tts-cache only.

1. Write `storage.tf` with the three buckets and their policies.
2. Grant `ui-sa` object-admin on uploads and tts-cache (not audit).

**Expected behaviour:** `gsutil lifecycle get` / `gsutil retention get` show the expected policies. Deleting a file from the audit bucket fails with a retention error.

In [ ]:
STORAGE_TF = '''
resource "google_storage_bucket" "uploads" {
  name          = "${var.project_id}-uploads"
  location      = var.india_region   # in-region residency (best practice, not a DPDP mandate)
  force_destroy = false
  uniform_bucket_level_access = true
  public_access_prevention    = "enforced"
  versioning { enabled = true }
  lifecycle_rule {
    condition { age = 90 }
    action    {
      type          = "SetStorageClass"
      storage_class = "NEARLINE"
    }
  }
  lifecycle_rule {
    condition { age = 365 }
    action    { type = "Delete" }
  }
  cors {
    origin          = ["https://documind.example.com"]
    method          = ["GET", "PUT", "POST"]
    response_header = ["Content-Type"]
    max_age_seconds = 3600
  }
}

resource "google_storage_bucket" "tts_cache" {
  name          = "${var.project_id}-tts-cache"
  location      = var.india_region
  uniform_bucket_level_access = true
  lifecycle_rule {
    condition { age = 30 }
    action    { type = "Delete" }
  }
}

resource "google_storage_bucket" "audit" {
  name          = "${var.project_id}-audit"
  location      = var.india_region
  uniform_bucket_level_access = true
  retention_policy {
    retention_period = 157680000   # 5 years -- audit/RBI retention best practice (DPDP Act sets no fixed number)
    is_locked        = true
  }
}

# UI can read/write uploads + TTS cache only
resource "google_storage_bucket_iam_member" "ui_uploads" {
  bucket = google_storage_bucket.uploads.name
  role   = "roles/storage.objectAdmin"
  member = "serviceAccount:${google_service_account.ui.email}"
}
resource "google_storage_bucket_iam_member" "ui_tts" {
  bucket = google_storage_bucket.tts_cache.name
  role   = "roles/storage.objectAdmin"
  member = "serviceAccount:${google_service_account.ui.email}"
}
'''
with open('storage.tf', 'w') as f: f.write(STORAGE_TF)
print('storage.tf written')

## Exercise 7: Secret Manager with 5 secrets + IAM scoping

**Difficulty:** Challenge

Create the 5 canonical secrets via `for_each`. Add `ui-sa` + `api-sa` as `secretAccessor` on each. Populate `cookie-secret` with `openssl rand -hex 32`. Verify `admin-sa` (which has no accessor binding) cannot read it.

1. Write `secrets.tf`: 5 secrets, auto-replication, accessor bindings for ui + api.
2. After apply, add a version to `cookie-secret`.
3. Confirm `ui-sa` can read the secret and `admin-sa` gets a 403.

**Expected behaviour:** accessing `cookie-secret` while impersonating `ui-sa` works; the same call impersonating `admin-sa` returns 403.

In [ ]:
SECRETS_TF = '''
variable "secret_names" {
  type    = set(string)
  default = [
    "litellm-master-key",
    "cookie-secret",
    "oauth-client-id",
    "oauth-client-secret",
    "openai-fallback-key",
  ]
}

resource "google_secret_manager_secret" "s" {
  for_each  = var.secret_names
  secret_id = each.value
  replication { auto {} }
}

resource "google_secret_manager_secret_iam_member" "ui_access" {
  for_each  = google_secret_manager_secret.s
  secret_id = each.value.id
  role      = "roles/secretmanager.secretAccessor"
  member    = "serviceAccount:${google_service_account.ui.email}"
}
resource "google_secret_manager_secret_iam_member" "api_access" {
  for_each  = google_secret_manager_secret.s
  secret_id = each.value.id
  role      = "roles/secretmanager.secretAccessor"
  member    = "serviceAccount:${google_service_account.api.email}"
}
'''
with open('secrets.tf', 'w') as f: f.write(SECRETS_TF)
print('secrets.tf written')

In [ ]:
%%bash
set -e
# Run AFTER `terraform apply`. Populate cookie-secret with a fresh random value.
echo -n "$(openssl rand -hex 32)" | gcloud secrets versions add cookie-secret --data-file=-

UI_SA="documind-ui-sa@${PROJECT_ID}.iam.gserviceaccount.com"
ADMIN_SA="documind-admin-sa@${PROJECT_ID}.iam.gserviceaccount.com"

echo '--- ui-sa SHOULD read the secret ---'
gcloud secrets versions access 1 --secret=cookie-secret \
  --impersonate-service-account="$UI_SA" && echo 'OK: ui-sa read the secret'

echo '--- admin-sa should be DENIED (403) ---'
gcloud secrets versions access 1 --secret=cookie-secret \
  --impersonate-service-account="$ADMIN_SA" \
  || echo 'Expected: admin-sa has no secretAccessor binding -> 403 PERMISSION_DENIED'

## Exercise 8: Billing budget with 4 thresholds + Pub/Sub

**Difficulty:** Challenge

Create a $500/month budget with thresholds 50/80/100 actual + 120 forecasted. Route notifications to Pub/Sub topic `documind-budget-alerts`. Write a short Cloud Function that drops Cloud Run `min_instances` to 0 on a 100% breach.

1. Write `budget.tf`: budget + 4 threshold rules + Pub/Sub topic.
2. Write the Cloud Function handler that degrades the service on breach.
3. Apply the whole stack and run the smoke tests.

**Expected behaviour:** the budget shows 4 threshold rules. Publishing a test message fires the function and sets `min_instances=0`.

In [ ]:
# $500/month budget. INR reference for the India team:
print(f'Budget cap: $500/month  ~=  Rs {500 * 85:,}/month  (USD_INR = 85)')

BUDGET_TF = '''
data "google_billing_account" "acct" {
  billing_account = var.billing_account_id
}

resource "google_billing_budget" "documind" {
  billing_account = data.google_billing_account.acct.id
  display_name    = "DocuMind monthly budget"

  budget_filter {
    projects = ["projects/${var.project_id}"]
  }

  amount {
    specified_amount {
    currency_code = "USD"
    units         = "500"
  }
  }

  threshold_rules { threshold_percent = 0.5 }
  threshold_rules { threshold_percent = 0.8 }
  threshold_rules { threshold_percent = 1.0 }
  threshold_rules {
    threshold_percent = 1.2
    spend_basis       = "FORECASTED_SPEND"
  }

  all_updates_rule {
    monitoring_notification_channels = var.alert_channels
    disable_default_iam_recipients   = false
    pubsub_topic                     = google_pubsub_topic.budget_alerts.id
  }
}

resource "google_pubsub_topic" "budget_alerts" {
  name = "documind-budget-alerts"
}

variable "billing_account_id" { type = string }
variable "alert_channels"     {
  type    = list(string)
  default = []
}
'''
with open('budget.tf', 'w') as f: f.write(BUDGET_TF)
print('budget.tf written')

In [ ]:
# Cloud Function (2nd gen, Pub/Sub-triggered): on a 100% actual breach,
# scale the Cloud Run service down to min_instances=0 to stop the bleed.
import os
os.makedirs('budget_guard', exist_ok=True)

MAIN_PY = '''
import base64, json, os
from google.cloud import run_v2

SERVICE = os.environ["GUARD_SERVICE"]          # projects/.../locations/.../services/documind-ui

def on_budget_alert(cloud_event):
    payload = json.loads(base64.b64decode(cloud_event.data["message"]["data"]))
    spent = payload.get("costAmount", 0)
    budget = payload.get("budgetAmount", 1)
    if spent < budget:                          # under 100% -> nothing to do
        return
    client = run_v2.ServicesClient()
    svc = client.get_service(name=SERVICE)
    svc.template.scaling.min_instance_count = 0  # kill always-on capacity
    client.update_service(service=svc)
    print(f"Budget breach {spent}/{budget}: {SERVICE} min_instances -> 0")
'''
with open('budget_guard/main.py', 'w') as f:
    f.write(MAIN_PY)
with open('budget_guard/requirements.txt', 'w') as f:
    f.write('google-cloud-run>=0.10.0\n')
print('budget_guard/main.py + requirements.txt written')

In [ ]:
%%bash
set -e
# Deploy the budget guard (run AFTER terraform apply creates the topic).
# NOTE: runs as documind-admin-sa, which MUST hold roles/run.developer (added in Exercise 3's
# admin_roles) -- run_v2.update_service needs run.services.update, or the guard hits PERMISSION_DENIED.
gcloud functions deploy budget-guard \
  --gen2 --runtime=python312 --region="$REGION" \
  --source=budget_guard --entry-point=on_budget_alert \
  --trigger-topic=documind-budget-alerts \
  --set-env-vars="GUARD_SERVICE=projects/${PROJECT_ID}/locations/${REGION}/services/documind-ui" \
  --service-account="documind-admin-sa@${PROJECT_ID}.iam.gserviceaccount.com"

# Smoke test: publish a fake 100%-breach message and confirm the function fires.
gcloud pubsub topics publish documind-budget-alerts \
  --message='{"costAmount":500,"budgetAmount":500,"currencyCode":"USD"}'
echo 'Check: gcloud functions logs read budget-guard --gen2 --region=$REGION'

In [ ]:
%%bash
set -e
# Apply the full stack, then smoke-test every resource created above.
terraform init -reconfigure
terraform plan -out=tfplan \
  -var=project_id="$PROJECT_ID" \
  -var=region="$REGION" \
  -var=india_region="$INDIA_REGION" \
  -var=billing_account_id=YOUR-BILLING-ID \
  -var='admin_emails=["alice@documind.ai"]'

terraform apply tfplan

echo '===== SMOKE TESTS ====='
gcloud iam service-accounts list --filter="email~documind-.*-sa"
gcloud artifacts repositories describe documind --location="$REGION"
gcloud firestore databases list
gsutil ls -b "gs://${PROJECT_ID}-uploads"
gcloud secrets list --filter=name~cookie-secret
gcloud billing budgets list --billing-account=YOUR-BILLING-ID